# TP6 : traces, métriques, alertes et diagnostic de lag

| Repère | Valeur |
|---|---|
| Moment | Jour 2, 11:30-12:30, après la démo du TP (11:25-11:30) |
| Compétences | C11 Traces log4j2 modifiées à chaud ; C12 Métriques, tableau de bord et alertes ; C13 Diagnostic d'un consumer lag |
| Slides de rappel | S53 (log4j2 à chaud), S54 et S55 (métriques broker et contrôleur), S57 (agent JMX), S58 (Prometheus et Grafana), S59 (alertes), S60 (lag) |
| Version Markdown | `enonce.md`, mêmes étapes en ligne de commande seule |

Mode requis : `extended`. En mode `core`, Prometheus, Grafana et le générateur de charge n'existent pas : ce TP est impossible.

**Mode d'emploi du notebook**

- Exécutez les cellules dans l'ordre avec `Maj+Entrée`. Après un redémarrage du noyau, les variables sont perdues : relancez la cellule d'outils, puis les cellules qui définissent les fonctions et les variables dont vous avez besoin. Ne relancez pas une cellule qui modifie le cluster (création d'un topic, arrêt d'un nœud) : elle échouerait, ou referait l'action.
- Chaque étape suit le même schéma : une explication, un **exemple en ligne de commande**, une **zone `# TODO`**, puis une **vérification**. Les commandes qui modifient le cluster sont affichées sans être exécutées (« ne pas lancer en plus du TODO »).
- Le notebook joint Prometheus et Grafana par leur nom dans le réseau du lab (`PROMETHEUS`, `GRAFANA`). Depuis votre navigateur, ce sont http://localhost:9090 et http://localhost:3000.
- Les étapes 6.7 et 6.8 attendent des alertes : leurs cellules restent 2 à 4 minutes en exécution (`[*]`). Ne les interrompez pas.

In [ ]:
# Connexion au lab : adresses des brokers, configuration du client Python, dossier de travail
import os
import time

from confluent_kafka.admin import AdminClient

BS = os.environ["BS"]  # listener INTERNAL, comme $BS dans le conteneur tools
PROMETHEUS = "http://prometheus:9090"
CONF = {"bootstrap.servers": BS, "log_level": 2}

# Les commandes docker compose et les fichiers config/ se lisent depuis le dossier lab/
LAB_DIR = os.environ["FORMATION_DIR"] + "/lab"
os.chdir(LAB_DIR)
os.environ["COMPOSE_PROGRESS"] = "plain"  # docker compose sans animation dans le notebook

In [ ]:
# Deux fonctions d'aide, ecrites ici : elles ne font pas partie de confluent-kafka
def attendre(condition, timeout=60, pas=2, message="condition"):
    """Rappelle condition() jusqu'a obtenir une valeur vraie, et la renvoie."""
    debut = time.time()
    while True:
        try:
            valeur = condition()
        except Exception:
            valeur = None
        if valeur:
            return valeur
        if time.time() - debut >= timeout:
            raise TimeoutError("%s : non atteinte en %d s" % (message, timeout))
        time.sleep(pas)

def tableau(lignes, entetes):
    """Affiche des lignes en tableau Markdown dans le notebook."""
    from IPython.display import Markdown, display

    md = ["| " + " | ".join(entetes) + " |", "|" + "---|" * len(entetes)]
    md += ["| " + " | ".join(str(v).replace("|", "\\|") for v in ligne) + " |" for ligne in lignes]
    display(Markdown("\n".join(md)))

In [ ]:
import os
import re
import shutil
import time

import requests
from confluent_kafka import ConsumerGroupTopicPartitions, KafkaException, TopicPartition
from confluent_kafka.admin import NewTopic, OffsetSpec

GRAFANA = "http://grafana:3000"

try:
    requests.get(PROMETHEUS + "/-/ready", timeout=3)
    print("Prometheus joignable : mode extended, TP possible")
except requests.RequestException:
    print("Prometheus absent : demarrez le lab en mode extended (scripts/start.sh extended)")

## 1. Objectifs

À la fin du TP, vous savez :

- retrouver où chaque famille de traces est écrite et avec quel niveau ;
- passer un logger en DEBUG sur un seul nœud, sans redémarrage, puis revenir au niveau normal ;
- lire les métriques brutes exposées par l'agent JMX d'un broker ;
- interroger Prometheus en PromQL et interpréter les indicateurs de santé ;
- relier le tableau de bord Grafana aux requêtes qui l'alimentent ;
- faire le ménage dans les consumer groups inactifs ;
- provoquer un lag, le suivre et voir une alerte passer de `pending` à `firing` ;
- écrire, valider et charger une règle d'alerte.

## 2. Prérequis

- Lab démarré en mode `extended` (supervision et générateur de charge) : `scripts/start.sh extended` puis `scripts/validate.sh`.
- TP du jour 1 et TP5 terminés. Les groupes `billing` et `billing-secure` existent.
- Navigateur sur l'hôte : Prometheus sur http://localhost:9090, Grafana sur http://localhost:3000.
- Notions : slides S52 à S60.

## 3. Architecture utilisée

```
  kafka-1..3 (JVM)                              loadgen-producer   loadgen-consumer
  MBeans -> agent JMX :9404/metrics             agent JMX client :9404/metrics
          |                                              |
          |            kafka-exporter :9308 (lag des groupes, lu dans le cluster)
          |                    |                         |
          +--------------------+-------------------------+
                               |  collecte toutes les 15 s
                               v
                Prometheus :9090  --  règles : monitoring/prometheus/rules/*.yml
                               |
                               v
                Grafana :3000  --  tableau de bord « Kafka - Cluster NovaShop (lab) »

  Traces : config/kafka/common/log4j2.yaml (monté sur chaque nœud)
           -> /opt/kafka/logs/server.log, controller.log, state-change.log,
              kafka-authorizer.log, log-cleaner.log, kafka-request.log
           niveau modifiable à chaud : kafka-configs.sh --entity-type broker-loggers
```

| Composant | Rôle dans le TP |
|---|---|
| `loadgen-producer` | Écrit 100 messages/s dans le topic `loadgen` |
| `loadgen-consumer` | Lit `loadgen` avec le groupe `loadgen-consumers` |
| `kafka-exporter` | Publie `kafka_consumergroup_lag`, calculé par le cluster |
| Règle `KafkaConsumerGroupLagHigh` | Lag d'un groupe sur un topic supérieur à 5000 pendant 1 minute (seuil de lab) |

| Lab | Production |
|---|---|
| Seuil de lag fixe de 5000 messages, atteint en moins d'une minute | Seuil exprimé en retard (secondes) ou en tendance, adapté à chaque application |
| Le fichier log4j2 est relu toutes les 30 s (`monitorInterval`, ajout de la formation) | Selon la politique de l'équipe : une relecture automatique permet aussi une erreur de saisie appliquée à chaud |

## 4. Contexte métier

L'équipe Facturation de NovaShop signale des factures émises avec plusieurs minutes de retard. L'exploitation dispose d'une chaîne de supervision complète, mais personne ne l'a encore prise en main. Votre mission : savoir augmenter les traces du contrôleur pendant un incident, retrouver les bons indicateurs, et prouver qu'une alerte prévient l'astreinte quand un consommateur s'arrête.

## 5. Énoncé

1. Identifiez les fichiers de traces et le logger du contrôleur.
2. Passez ce logger en DEBUG sur le seul contrôleur actif, observez l'effet, puis revenez au niveau normal.
3. Retrouvez dans les métriques brutes, puis dans Prometheus et Grafana, les indicateurs de santé du cluster.
4. Supprimez les consumer groups qui ne servent plus.
5. Arrêtez le consommateur du générateur de charge, suivez le lag et l'alerte, puis rétablissez le service.
6. Écrivez une alerte qui détecte l'arrêt du producteur, prouvez qu'elle fonctionne, puis retirez-la.

## 6. Étapes

### 6.1 Où vont les traces

**Exemples en ligne de commande** : la configuration des traces, puis les fichiers d'un nœud.

In [ ]:
!grep -nE 'monitorInterval|fileName' config/kafka/common/log4j2.yaml
!grep -n -A5 'name: org.apache.kafka.controller' config/kafka/common/log4j2.yaml
!docker compose exec -T kafka-1 ls /opt/kafka/logs </dev/null

In [ ]:
# TODO 6.1 : lire le fichier log4j2 en Python
# Le fichier est lisible depuis le notebook : os.path.join(LAB_DIR, "config/kafka/common/log4j2.yaml").
# 1. intervalle : la valeur de monitorInterval, en entier (re.search).
# 2. fichiers_traces : la liste des noms de fichiers declares, par exemple "server.log" (re.findall sur les
#    lignes fileName: "${sys:kafka.logs.dir}/server.log").
intervalle = None
fichiers_traces = []

print("relecture toutes les", intervalle, "s ;", fichiers_traces)

In [ ]:
# Verification 6.1
assert intervalle == 30, "A revoir : fichier relu toutes les 30 s"
print("OK  fichier relu toutes les 30 s")
assert len(fichiers_traces) == 6 and "controller.log" in fichiers_traces, "A revoir : 6 fichiers de traces, dont controller.log"
print("OK  6 fichiers de traces, dont controller.log")

**Résultat attendu** :

- `monitorInterval: 30` : le fichier est relu toutes les 30 s ;
- 6 fichiers déclarés : `server.log`, `state-change.log`, `kafka-request.log`, `log-cleaner.log`, `controller.log`, `kafka-authorizer.log` ;
- `org.apache.kafka.controller` : niveau `INFO`, `additivity: false`, appender `ControllerAppender`. Ses lignes vont donc dans `controller.log` uniquement ;
- les fichiers tournent toutes les heures (`controller.log.2026-10-28-15` contient l'heure 15, en UTC) ; on trouve aussi les traces du ramasse-miettes (`kafkaServer-gc.log`).

### 6.2 Changer un niveau de trace à chaud

Le niveau d'un logger se change sans redémarrage, nœud par nœud, avec `kafka-configs.sh --entity-type broker-loggers`. Le client Python n'a pas d'API pour cela (`ConfigResource` ne connaît pas ce type) : dans cette étape, c'est une cellule `!docker exec kafka-tools kafka-configs.sh ...` qui appelle l'outil Java.

**Exemples en ligne de commande** : le contrôleur actif (`LeaderId`), le nombre de loggers configurables sur un nœud et le niveau de deux d'entre eux.

In [ ]:
!docker exec kafka-tools kafka-metadata-quorum.sh --bootstrap-server $BS describe --status | grep LeaderId
!docker exec kafka-tools bash -c "kafka-configs.sh --bootstrap-server $BS --describe --entity-type broker-loggers --entity-name 1 | grep -c '='"
!docker exec kafka-tools bash -c "kafka-configs.sh --bootstrap-server $BS --describe --entity-type broker-loggers --entity-name 1 | grep -E ' (org.apache.kafka.controller|root)='"

La fonction `lignes_debug(noeud)` compte les lignes DEBUG de `controller.log` sur un nœud.

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**), pour le contrôleur actif `$L` :

```bash
# [tools]
kafka-configs.sh --bootstrap-server $BS --alter --entity-type broker-loggers --entity-name $L \
  --add-config org.apache.kafka.controller=DEBUG
kafka-configs.sh --bootstrap-server $BS --alter --entity-type broker-loggers --entity-name $L \
  --add-config kafka.controller=DEBUG
kafka-topics.sh --bootstrap-server $BS --create --topic inventory --partitions 3 --replication-factor 3 \
  --config cleanup.policy=compact
```

In [ ]:
def lignes_debug(noeud):
    sortie = !docker compose exec -T kafka-{noeud} sh -c 'grep -c DEBUG /opt/kafka/logs/controller.log' </dev/null 2>&1
    sortie = sortie.n
    return int(sortie.split()[-1])

In [ ]:
# TODO 6.2 a : passer le logger du controleur en DEBUG sur le seul controleur actif
# 1. leader : le LeaderId, en entier, lu dans la sortie de `sortie = !docker exec kafka-tools kafka-metadata-quorum.sh ... describe --status`
#    (joignez les lignes avec sortie.n) avec re.search.
# 2. Avec `!docker exec kafka-tools ...`, passez org.apache.kafka.controller en DEBUG sur ce noeud (--entity-name leader).
# 3. Essayez le nom kafka.controller, courant dans les documentations 3.x : rangez la sortie dans refus_ancien_nom.
# 4. Creez le topic inventory en Python : 3 partitions, RF 3, config={"cleanup.policy": "compact"}.
# 5. debug_en_10s : augmentation de lignes_debug(leader) en 10 s (time.sleep(10)).
# 6. creation_tracee : lignes de controller.log du leader qui contiennent inventory, avec `!docker compose exec -T kafka-N ...` et grep
#    (sur le modele de lignes_debug ; ajoutez | cut -c1-200 pour couper les lignes).
leader = None
refus_ancien_nom = creation_tracee = ""
debug_en_10s = None

print("controleur actif :", leader, "- lignes DEBUG en 10 s :", debug_en_10s)

In [ ]:
# Verification 6.2 a
assert leader in (1, 2, 3), "A revoir : controleur actif : kafka-%s" % leader
print("OK  controleur actif : kafka-%s" % leader)
assert "Invalid broker logger" in refus_ancien_nom, "A revoir : le nom 3.x kafka.controller est refuse"
print("OK  le nom 3.x kafka.controller est refuse")
assert debug_en_10s is not None and debug_en_10s > 20, "A revoir : %s lignes DEBUG en 10 s" % debug_en_10s
print("OK  %s lignes DEBUG en 10 s" % debug_en_10s)
assert "CreateTopics result" in creation_tracee and "inventory" in creation_tracee, "A revoir : creation d'inventory tracee par le controleur"
print("OK  creation d'inventory tracee par le controleur")

Une ligne DEBUG typique :

In [ ]:
!docker compose exec -T kafka-{leader} sh -c 'grep DEBUG /opt/kafka/logs/controller.log | tail -3' </dev/null | cut -c1-220

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-configs.sh --bootstrap-server $BS --alter --entity-type broker-loggers --entity-name $L \
  --delete-config org.apache.kafka.controller
```

In [ ]:
# TODO 6.2 b : revenir au niveau normal
# 1. Avec `!docker exec kafka-tools ...`, retirez le reglage (--delete-config org.apache.kafka.controller) sur le leader.
# 2. niveau_final : la ligne " org.apache.kafka.controller=" du --describe des broker-loggers du leader (grep).
# 3. debug_apres : augmentation de lignes_debug(leader) en 10 s.
niveau_final = ""
debug_apres = None

print(niveau_final, "- lignes DEBUG en 10 s :", debug_apres)

In [ ]:
# Verification 6.2 b
assert "org.apache.kafka.controller=INFO" in niveau_final, "A revoir : logger du controleur revenu a INFO"
print("OK  logger du controleur revenu a INFO")
assert debug_apres == 0, "A revoir : plus aucune ligne DEBUG"
print("OK  plus aucune ligne DEBUG")

**Résultat attendu** :

- environ 200 loggers configurables sur un nœud ; `org.apache.kafka.controller` et `root` en `INFO` ;
- le passage en DEBUG répond `Completed updating config for broker-logger <n>.` ; `kafka.controller` est refusé : `Invalid broker logger(s): kafka.controller` ;
- de quelques dizaines à quelques centaines de lignes DEBUG en 10 s, même sans activité d'administration ;
- la création d'`inventory` apparaît en INFO : `CreateTopics result(s): CreatableTopic(name='inventory', ...): SUCCESS`, puis `Replayed TopicRecord`, `Replayed ConfigRecord` et un `Replayed PartitionRecord` par partition ;
- après `--delete-config`, le logger repasse en `INFO` et le compteur DEBUG ne bouge plus.

### 6.3 Lire les métriques brutes

L'agent JMX de chaque nœud publie ses métriques en texte sur le port 9404, une série par ligne : `nom{etiquettes} valeur`. Les lignes qui commencent par `#` décrivent la métrique (`HELP`, `TYPE`).

**Exemple en ligne de commande** : trois métriques de `kafka-1`.

In [ ]:
!docker exec kafka-tools bash -c "wget -qO- http://kafka-1:9404/metrics | grep -E '^kafka_server_replicamanager_(underreplicatedpartitions|leadercount|partitioncount)'"

In [ ]:
# TODO 6.3 : les metriques brutes en Python
# 1. Ecrivez metriques(noeud) : elle lit requests.get("http://kafka-%d:9404/metrics" % noeud, timeout=10).text
#    et renvoie un dictionnaire {serie: valeur en float}, sans les lignes vides ni celles qui commencent par #.
#    La valeur est le dernier mot de la ligne : ligne.rsplit(" ", 1).
# 2. nb_series : nombre de series exposees par kafka-1.
# 3. replication_1 : pour kafka-1, les valeurs de kafka_server_replicamanager_leadercount, ..._partitioncount
#    et ..._underreplicatedpartitions, dans un dictionnaire {nom: valeur}.
# 4. controleurs_actifs : {noeud: valeur de kafka_controller_kafkacontroller_activecontrollercount} pour 1, 2 et 3.
def metriques(noeud):
    return {}


nb_series = None
replication_1 = controleurs_actifs = {}

print(nb_series, "series ;", replication_1)
print(controleurs_actifs)

In [ ]:
# Verification 6.3
assert nb_series is not None and nb_series > 1000, "A revoir : %s series exposees par kafka-1" % nb_series
print("OK  %s series exposees par kafka-1" % nb_series)
assert replication_1.get("underreplicatedpartitions") == 0.0, "A revoir : aucune partition sous-repliquee sur kafka-1"
print("OK  aucune partition sous-repliquee sur kafka-1")
assert sum(controleurs_actifs.values()) == 1 and controleurs_actifs.get(leader) == 1.0, "A revoir : un seul controleur actif, le meme qu'en 6.2 (kafka-%s)" % leader
print("OK  un seul controleur actif, le meme qu'en 6.2 (kafka-%s)" % leader)

**Résultat attendu** : quelques milliers de séries par nœud ; `underreplicatedpartitions` à 0 ; `activecontrollercount` vaut 1 sur le contrôleur actif trouvé en 6.2 et 0 sur les deux autres.

### 6.4 Interroger Prometheus

Prometheus collecte ces séries toutes les 15 s et répond aux requêtes PromQL par son API : `GET /api/v1/query?query=...`. La réponse JSON contient `data.result`, une liste d'éléments avec `metric` (les étiquettes) et `value` (`[horodatage, "valeur"]`, la valeur étant une chaîne).

**Exemple en ligne de commande** : une requête, depuis `tools`.

In [ ]:
!docker exec kafka-tools bash -c "wget -qO- --post-data 'query=sum(kafka_controller_kafkacontroller_activecontrollercount)' http://prometheus:9090/api/v1/query; echo"

In [ ]:
# TODO 6.4 : PromQL en Python
# 1. Ecrivez promql(requete) : liste de (etiquettes, valeur en float), a partir de
#    requests.get(PROMETHEUS + "/api/v1/query", params={"query": requete}, timeout=10).json().
# 2. Avec promql(), rangez :
#    controleurs  : sum(kafka_controller_kafkacontroller_activecontrollercount)          (un nombre)
#    urp          : sum(kafka_server_replicamanager_underreplicatedpartitions)           (un nombre)
#    debit        : {topic: valeur} pour sum by (topic) (rate(kafka_server_brokertopicmetrics_messagesin_total{topic!=""}[1m]))
#    inactivite   : {instance: valeur} pour kafka_server_kafkarequesthandlerpool_requesthandleravgidlepercent
#    lags         : {(consumergroup, topic): valeur} pour sum by (consumergroup, topic) (kafka_consumergroup_lag)
#    cibles       : {(job, instance): valeur} pour up
def promql(requete):
    return []


controleurs = urp = None
debit = inactivite = lags = cibles = {}

tableau([("controleurs actifs", controleurs), ("partitions sous-repliquees", urp),
         ("debit par topic", {t: round(v, 1) for t, v in debit.items() if v > 0}),
         ("inactivite des threads", {i: round(v, 3) for i, v in inactivite.items()}),
         ("lag par groupe", lags), ("cibles a 0", [c for c, v in cibles.items() if v == 0])], ["Indicateur", "Valeur"])

In [ ]:
# Verification 6.4
assert controleurs == 1 and urp == 0, "A revoir : 1 controleur actif, 0 partition sous-repliquee"
print("OK  1 controleur actif, 0 partition sous-repliquee")
assert 50 < debit.get("loadgen", 0) < 150, "A revoir : debit de loadgen : %s messages/s" % round(debit.get("loadgen", 0), 1)
print("OK  debit de loadgen : %s messages/s" % round(debit.get("loadgen", 0), 1))
assert len(inactivite) == 3 and min(inactivite.values()) > 0.5, "A revoir : threads de requetes peu occupes"
print("OK  threads de requetes peu occupes")
assert ("loadgen-consumers", "loadgen") in lags, "A revoir : lag de loadgen-consumers publie par kafka-exporter"
print("OK  lag de loadgen-consumers publie par kafka-exporter")
assert [c for c, v in cibles.items() if v == 0] == [("kafka-scale", "kafka-4")], "A revoir : seule la cible kafka-4 est a 0"
print("OK  seule la cible kafka-4 est a 0")

**Résultat attendu** :

| Indicateur | Valeur attendue |
|---|---|
| Contrôleurs actifs | 1 |
| Partitions sous-répliquées | 0 |
| Débit par topic | `loadgen` vers 100 messages/s, environ 1 pour `__consumer_offsets`, 0 pour les topics inactifs |
| Inactivité des threads | Environ 0,99 sur chaque nœud |
| Lag par groupe | Quelques centaines au plus pour `loadgen-consumers`, valeurs fixes pour les groupes arrêtés |
| `up` | 1 pour toutes les cibles, sauf `kafka-4` (job `kafka-scale`) à 0 |

Dans le navigateur : http://localhost:9090, onglet Graph, requête `sum by (topic) (rate(kafka_server_brokertopicmetrics_messagesin_total{topic!=""}[1m]))`, vue Graph sur 15 minutes.

### 6.5 Lire le tableau de bord Grafana

Ouvrez http://localhost:3000, menu Dashboards, puis « Kafka - Cluster NovaShop (lab) », période « Last 15 minutes ». La consultation est ouverte sans compte.

Un tableau de bord Grafana est un document JSON : chaque panneau porte sa requête PromQL. L'API de Grafana le renvoie : `GET /api/dashboards/uid/kafka-cluster`. Dans `dashboard["panels"]`, les panneaux se suivent dans l'ordre d'affichage ; un panneau de type `row` ouvre une rangée, et la requête d'un panneau est dans `panneau["targets"][0]["expr"]`.

**Exemple en ligne de commande** : les titres des panneaux, depuis `tools`.

In [ ]:
!docker exec kafka-tools bash -c 'wget -qO- http://grafana:3000/api/dashboards/uid/kafka-cluster | grep -o '"'"'"title":"[^"]*"'"'"' | head -12'

In [ ]:
# TODO 6.5 : recalculer la rangee "Sante du cluster"
# 1. panneaux = requests.get(GRAFANA + "/api/dashboards/uid/kafka-cluster", timeout=10).json()["dashboard"]["panels"]
# 2. sante : liste de (titre, valeur) pour les panneaux situes apres la rangee (type "row") titree
#    "Sante du cluster" et avant la rangee suivante. Valeur : promql(expr)[0][1].
sante = []

tableau(sante, ["Panneau", "Valeur"])

In [ ]:
# Verification 6.5
valeurs_sante = dict(sante)
assert len(sante) == 8, "A revoir : 8 panneaux dans la rangee Sante du cluster"
print("OK  8 panneaux dans la rangee Sante du cluster")
assert valeurs_sante.get("Noeuds joignables") == 3 and valeurs_sante.get("Brokers actifs") == 3, "A revoir : 3 noeuds, 3 brokers actifs"
print("OK  3 noeuds, 3 brokers actifs")
assert valeurs_sante.get("Leader du quorum (node.id)") == leader, "A revoir : leader du quorum : kafka-%s, comme en 6.2" % leader
print("OK  leader du quorum : kafka-%s, comme en 6.2" % leader)
assert (all(valeurs_sante.get(t) == 0 for t in ("Brokers isoles (fenced)", "Partitions sous-repliquees",
                                                "Sous min ISR", "Partitions offline"))), "A revoir : aucun indicateur de panne"
print("OK  aucun indicateur de panne")

**Résultat attendu** : les 8 valeurs sont celles des tuiles de la rangée « Sante du cluster » dans le navigateur. Dans les autres rangées, « Messages entrants / s par topic » montre `loadgen` vers 100 et « Leaders par broker » une répartition à peu près égale.

### 6.6 Nettoyer les groupes inactifs

**Exemple en ligne de commande** : les groupes et leur état.

In [ ]:
!docker exec kafka-tools kafka-consumer-groups.sh --bootstrap-server $BS --list --state

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-consumer-groups.sh --bootstrap-server $BS --delete --group loadgen-consumers
kafka-consumer-groups.sh --bootstrap-server $BS --delete --group analytics --group analytics-v2
```

In [ ]:
# TODO 6.6 : supprimer les groupes inactifs
# 1. Essayez de supprimer loadgen-consumers : admin.delete_consumer_groups([...])[groupe].result() ;
#    rangez le code d'erreur de la KafkaException dans refus_suppression.
# 2. a_supprimer : analytics, analytics-v2 et les groupes dont le nom commence par console-consumer-,
#    parmi ceux qui existent (admin.list_consumer_groups().result().valid, attribut group_id).
#    Supprimez-les (attendez chaque futur ; ne rien appeler si la liste est vide).
# 3. groupes_restants : liste triee des groupes apres le menage.
refus_suppression = None
a_supprimer = groupes_restants = []

print("refus :", refus_suppression, "- supprimes :", a_supprimer, "- restants :", groupes_restants)

In [ ]:
# Verification 6.6
assert refus_suppression == "NON_EMPTY_GROUP", "A revoir : suppression d'un groupe actif refusee"
print("OK  suppression d'un groupe actif refusee")
assert not [g for g in groupes_restants if g.startswith(("analytics", "console-consumer-"))], "A revoir : groupes inactifs supprimes"
print("OK  groupes inactifs supprimes")
assert {"billing", "billing-secure", "loadgen-consumers"} <= set(groupes_restants), "A revoir : billing, billing-secure et loadgen-consumers conserves"
print("OK  billing, billing-secure et loadgen-consumers conserves")

**Résultat attendu** : `loadgen-consumers` est `STABLE` ; sa suppression est refusée (`NON_EMPTY_GROUP`, `GroupNotEmptyException` côté Java). Restent `billing`, `billing-secure` et `loadgen-consumers`.

### 6.7 Provoquer et diagnostiquer un lag

Le **lag** d'un groupe est la somme, sur ses partitions, de l'écart entre la fin de la partition et l'offset validé par le groupe.

La fonction `suivre(nombre)` ci-dessous relève, toutes les 15 s, le lag de `loadgen-consumers` et l'état des alertes, avec deux fonctions que vous écrivez dans le `TODO` a. Les temps sont comptés depuis le lancement de la cellule.

In [ ]:
def suivre(nombre, pas=15, groupe="loadgen-consumers"):
    debut, releves = time.time(), []
    for _ in range(nombre):
        time.sleep(pas)
        releve = (round(time.time() - debut), lag_groupe(groupe), etats_alertes())
        releves.append(releve)
        print("t+%3d s  lag=%6d  alertes=%s" % releve)
    return releves

In [ ]:
# TODO 6.7 a : les deux fonctions de suivi
# 1. lag_groupe(groupe) : offsets valides par
#      admin.list_consumer_group_offsets([ConsumerGroupTopicPartitions(groupe)])[groupe].result().topic_partitions
#    (ignorer les offsets negatifs), fins de partition par admin.list_offsets({TopicPartition(...): OffsetSpec.latest()}),
#    puis la somme des ecarts.
# 2. etats_alertes() : {nom de l'alerte: etat} a partir de requests.get(PROMETHEUS + "/api/v1/alerts", timeout=10)
#    .json()["data"]["alerts"] ; chaque alerte a "labels" (dont "alertname") et "state" ("pending" ou "firing").
def lag_groupe(groupe):
    return 0


def etats_alertes():
    return {}


print("lag :", lag_groupe("loadgen-consumers"), "- alertes :", etats_alertes())

Équivalent en ligne de commande du `TODO` b (**ne pas lancer en plus du TODO**) :

```bash
# [hôte]
docker compose stop loadgen-consumer
# [tools] une fois l'alerte en firing
kafka-consumer-groups.sh --bootstrap-server $BS --describe --group loadgen-consumers
```

**Exécutez la cellule `TODO` b une seule fois** : elle dure environ 3 minutes. Avant de la lancer, notez votre prédiction : au bout de combien de temps l'alerte `KafkaConsumerGroupLagHigh` passe-t-elle en `firing` ?

In [ ]:
# TODO 6.7 b : arreter le consommateur et suivre
# 1. Arretez loadgen-consumer : `!docker compose stop loadgen-consumer`.
# 2. releves_arret = suivre(12)
# 3. groupe_arrete : admin.describe_consumer_groups(["loadgen-consumers"])["loadgen-consumers"].result() ;
#    rangez dans etat_arrete le tuple (nom de l'etat, nombre de membres).
releves_arret = []
etat_arrete = None

print("etat du groupe :", etat_arrete)

In [ ]:
# Verification 6.7 b
assert releves_arret and max(r[1] for r in releves_arret) > 5000, "A revoir : lag au-dela de 5000"
print("OK  lag au-dela de 5000")
assert any(r[2].get("KafkaConsumerGroupLagHigh") == "firing" for r in releves_arret), "A revoir : alerte KafkaConsumerGroupLagHigh en firing"
print("OK  alerte KafkaConsumerGroupLagHigh en firing")
assert etat_arrete == ("EMPTY", 0), "A revoir : groupe sans membre : %s" % (etat_arrete,)
print("OK  groupe sans membre : %s" % (etat_arrete,))

Les métriques du consommateur lui-même ont disparu avec lui : sa cible est à 0 et sa métrique de lag n'existe plus.

In [ ]:
print("up du consommateur :", promql('up{instance="loadgen-consumer"}'))
print("records_lag_max :", promql("count(kafka_consumer_consumer_fetch_manager_metrics_records_lag_max)"))

In [ ]:
# TODO 6.7 c : redemarrer le consommateur et mesurer le rattrapage
# 1. `!docker compose start loadgen-consumer`
# 2. releves_reprise = suivre(6)
releves_reprise = []

In [ ]:
# Verification 6.7 c
assert releves_reprise and releves_reprise[-1][1] < 1000, "A revoir : lag rattrape"
print("OK  lag rattrape")
assert releves_reprise and "KafkaConsumerGroupLagHigh" not in releves_reprise[-1][2], "A revoir : alerte terminee"
print("OK  alerte terminee")

**Résultat attendu** :

| Temps après l'arrêt | Lag | Alerte `KafkaConsumerGroupLagHigh` |
|---|---|---|
| 15 s | Quelques centaines à 1500 | Aucune |
| 1 min | Environ 5000 à 6000 | Aucune ou `pending` |
| 2 min à 2 min 30 | Environ 12000 à 15000 | `firing` |

- Le lag augmente d'environ 1500 toutes les 15 s, soit le débit du producteur ;
- pendant l'arrêt, le groupe est `EMPTY`, sans membre ;
- après le redémarrage, le lag retombe sous 1000 en moins d'une minute, et l'alerte disparaît.

### 6.8 Défi : écrire une alerte

L'alerte `KafkaLoadgenNoTraffic` doit se déclencher quand le topic `loadgen` reçoit moins de 10 messages par seconde, tous brokers confondus, pendant 1 minute. Prometheus lit ses règles dans `monitoring/prometheus/rules/`, un dossier du lab que le notebook peut écrire : `FICHIER_REGLE` ci-dessous.

La fonction `suivre_alerte(nom)` relève l'état d'une alerte toutes les 15 s, jusqu'à `firing` (4 minutes au plus).

In [ ]:
FICHIER_REGLE = os.path.join(LAB_DIR, "monitoring/prometheus/rules/tp6-custom.yml")

REGLE_MODELE = """groups:
  - name: tp6-custom
    rules:
      - alert: KafkaLoadgenNoTraffic
        expr: <requete PromQL : messages entrants par seconde sur le topic loadgen, tous brokers confondus, inferieurs a 10>
        for: <duree>
        labels:
          severity: warning
        annotations:
          summary: "Le topic loadgen recoit moins de 10 messages/s depuis 1 minute"
"""


def suivre_alerte(nom, nombre=16, pas=15):
    debut, releves = time.time(), []
    for _ in range(nombre):
        time.sleep(pas)
        releves.append((round(time.time() - debut), etats_alertes().get(nom, "absente")))
        print("t+%3d s  %s : %s" % (releves[-1][0], nom, releves[-1][1]))
        if releves[-1][1] == "firing":
            break
    return releves

Équivalent en ligne de commande du `TODO` a (**ne pas lancer en plus du TODO**) :

```bash
# [hôte] après avoir écrit monitoring/prometheus/rules/tp6-custom.yml
docker compose exec prometheus promtool check rules /etc/prometheus/rules/tp6-custom.yml
curl -s -w 'HTTP %{http_code}\n' -X POST localhost:9090/-/reload
```

In [ ]:
# TODO 6.8 a : ecrire, valider et charger la regle
# 1. regle : REGLE_MODELE dont vous remplacez les deux valeurs entre < > (expression PromQL et duree).
# 2. Ecrivez regle dans FICHIER_REGLE (open(..., "w")).
# 3. controle : `!docker compose exec -T prometheus promtool check rules /etc/prometheus/rules/tp6-custom.yml`
# 4. code_rechargement : code HTTP de requests.post(PROMETHEUS + "/-/reload", timeout=10).
# 5. regles_chargees : noms des regles de requests.get(PROMETHEUS + "/api/v1/rules", timeout=10).json()["data"]["groups"]
#    (chaque groupe a "rules", chaque regle a "name").
regle = REGLE_MODELE
controle = ""
code_rechargement = None
regles_chargees = []

print("rechargement :", code_rechargement, "- KafkaLoadgenNoTraffic chargee :", "KafkaLoadgenNoTraffic" in regles_chargees)

In [ ]:
# Verification 6.8 a
assert "SUCCESS: 1 rules found" in controle, "A revoir : regle valide pour promtool"
print("OK  regle valide pour promtool")
assert code_rechargement == 200 and "KafkaLoadgenNoTraffic" in regles_chargees, "A revoir : regle chargee par Prometheus"
print("OK  regle chargee par Prometheus")

**Exécutez la cellule `TODO` b une seule fois** : elle dure 2 à 3 minutes.

In [ ]:
# TODO 6.8 b : prouver que l'alerte fonctionne
# 1. Arretez loadgen-producer (hote).
# 2. releves_defi = suivre_alerte("KafkaLoadgenNoTraffic")
# 3. Redemarrez loadgen-producer.
releves_defi = []

In [ ]:
# Verification 6.8 b
assert releves_defi and releves_defi[-1][1] == "firing", "A revoir : KafkaLoadgenNoTraffic en firing a t+%s s" % (releves_defi[-1][0] if releves_defi else "?")
print("OK  KafkaLoadgenNoTraffic en firing a t+%s s" % (releves_defi[-1][0] if releves_defi else "?"))

**Résultat attendu** : `SUCCESS: 1 rules found`, rechargement `200`. Après l'arrêt du producteur : alerte absente pendant 45 s environ, `pending` vers 1 min, `firing` vers 2 min.

### 6.9 Remettre le lab en état

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [hôte]
rm monitoring/prometheus/rules/tp6-custom.yml
curl -s -w 'HTTP %{http_code}\n' -X POST localhost:9090/-/reload
docker compose ps loadgen-producer loadgen-consumer
```

In [ ]:
# TODO 6.9 : remise en etat
# 1. Supprimez FICHIER_REGLE (os.remove) et rechargez Prometheus.
# 2. Attendez (attendre, timeout=180, pas=10) que le debit de loadgen depasse 50 messages/s et qu'aucune
#    alerte ne soit active.
# 3. clients_charge : `clients_charge = !docker compose ps --format '{{{{.Name}}}} {{{{.State}}}}' loadgen-producer loadgen-consumer`
clients_charge = ""

print(clients_charge)

In [ ]:
# Verification 6.9
assert not os.path.exists(FICHIER_REGLE), "A revoir : regle personnalisee retiree"
print("OK  regle personnalisee retiree")
assert clients_charge.count("running") == 2, "A revoir : les deux clients du generateur tournent"
print("OK  les deux clients du generateur tournent")
assert not etats_alertes(), "A revoir : aucune alerte active"
print("OK  aucune alerte active")
sortie = !docker exec kafka-tools bash -c "kafka-configs.sh --bootstrap-server $BS --describe --entity-type broker-loggers --entity-name {leader}" 2>&1
assert ("org.apache.kafka.controller=INFO" in sortie.n), "A revoir : logger du controleur en INFO"
print("OK  logger du controleur en INFO")

## Bilan de l'exercice

In [ ]:
tableau([
    ("Controleur actif", "kafka-%s" % leader, "6.2"),
    ("Lignes DEBUG en 10 s, puis apres retour a INFO", "%s, puis %s" % (debug_en_10s, debug_apres), "6.2"),
    ("Series exposees par kafka-1", nb_series, "6.3"),
    ("Debit de loadgen (messages/s)", round(debit.get("loadgen", 0), 1) if debit else "a completer", "6.4"),
    ("Groupes supprimes", ", ".join(a_supprimer) or "aucun", "6.6"),
    ("Premier releve en firing (lag)", next(("t+%d s" % r[0] for r in releves_arret
                                             if r[2].get("KafkaConsumerGroupLagHigh") == "firing"), "a completer"), "6.7"),
    ("KafkaLoadgenNoTraffic en firing", "t+%d s" % releves_defi[-1][0] if releves_defi else "a completer", "6.8"),
], ["Controle", "Valeur", "Etape"])

## 7. Questions de réflexion

1. Pourquoi passer le logger en DEBUG sur le seul contrôleur actif ? Que perdez-vous si le leadership du quorum change pendant l'incident ?
2. Le niveau modifié par `broker-loggers` survit-il au redémarrage du nœud ? Et une modification de `log4j2.yaml` ?
3. Que signifie une somme de `activecontrollercount` égale à 0 ? Égale à 2 ?
4. `kafka-4` est à `up=0`. Pourquoi n'est-ce pas une alerte ?
5. `requesthandleravgidlepercent` vaut 0,99. Que signifierait une valeur de 0,2, et quelle alerte se déclencherait ?
6. Le lag dépasse 5000 environ 1 minute après l'arrêt, mais l'alerte ne passe en `firing` qu'une minute plus tard. Quel compromis règle la durée `for` ?
7. Le consommateur arrêté, sa métrique `records_lag_max` disparaît. Pourquoi le lag doit-il être mesuré côté cluster ?
8. Un lag de 5000 messages est-il grave ? Comment l'exprimer en temps de retard ?
9. Quel risque prenez-vous en supprimant un groupe vide qui appartient à une application simplement arrêtée ?